In [1]:
import os
import numpy as np
import tensorflow as tf

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, _, _ = load_cache()
# Lead I model: per-window z-score (see project.CNN_INPUT_NORM); 3-lead: raw mV
X_train, X_test = normalize_cnn_input(X_train), normalize_cnn_input(X_test)
print(f"CNN input normalisation: {CNN_INPUT_NORM}")

print("Loading saved 1D-CNN model weights...")
cnn_1d = tf.keras.models.load_model(MODELS_DIR / "tinyml_1d_cnn.keras")

def representative_data_gen():
    # Use 200 calibration windows from training set to determine INT8 scaling
    for i in range(200):
        yield [X_train[i:i+1].astype(np.float32)]

print("Converting and Quantizing model to INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(cnn_1d)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Force strict integer precision
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_path = MODELS_DIR / "tinyml_1d_cnn_int8.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_quant_model)

raw_size = os.path.getsize(tflite_path) / 1024.0
print(f"Quantized Model Size: {raw_size:.2f} KB")

# Generate the C-array header for TensorFlow Lite Micro
header_path = MODELS_DIR / "model_data.h"
in_q = tf.lite.Interpreter(model_content=tflite_quant_model).get_input_details()[0]["quantization"]
with open(header_path, "w") as f:
    f.write(f"// Auto-generated TFLM Model Header\n")
    f.write(f"// Lead set: {LEAD_SET} (channels {LEADS})\n")
    f.write(f"// Input normalisation before quantising: {CNN_INPUT_NORM}"
            f"{' (x - mean) / (std + ' + str(ZSCORE_EPS_MV) + ' mV) per window' if CNN_INPUT_NORM == 'zscore' else ' (raw mV)'}\n")
    f.write(f"// q = round(x / g_model_input_scale) + g_model_input_zero_point\n")
    f.write(f"#ifndef ECG_MODEL_DATA_H_\n#define ECG_MODEL_DATA_H_\n\n")
    f.write(f"const float g_model_input_scale = {in_q[0]:.9g}f;\n")
    f.write(f"const int g_model_input_zero_point = {in_q[1]};\n")
    f.write(f"const unsigned int g_model_len = {len(tflite_quant_model)};\n")
    f.write(f"const unsigned char g_model[] __attribute__((aligned(4))) = {{\n  ")
    for i, b in enumerate(tflite_quant_model):
        f.write(f"0x{b:02x}, ")
        if (i + 1) % 12 == 0:
            f.write("\n  ")
    f.write("\n};\n\n#endif  // ECG_MODEL_DATA_H_\n")

print(f"Generated embedded C-header file: '{header_path}'")

Lead set  -> lead1 (channels [0])
Signals   -> Train: (19230, 1000, 1) | Test: (2158, 1000, 1)
Features  -> Train: (19230, 24) | Test: (2158, 24)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']
CNN input normalisation: zscore
Loading saved 1D-CNN model weights...


Converting and Quantizing model to INT8...


INFO:tensorflow:Assets written to: C:\Users\Asus\AppData\Local\Temp\tmpjr8pqiup\assets


INFO:tensorflow:Assets written to: C:\Users\Asus\AppData\Local\Temp\tmpjr8pqiup\assets


Saved artifact at 'C:\Users\Asus\AppData\Local\Temp\tmpjr8pqiup'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1000, 1), dtype=tf.float32, name='ecg_input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1949059949840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059951760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059950992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059951376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059949648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059951184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059952336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059951568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059953296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059953104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1949059952912

Quantized Model Size: 24.59 KB
Generated embedded C-header file: 'E:\ML\MTP\models\lead1\model_data.h'


C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [3]:
# Accuracy of the INT8 model actually deployed (the float AUC alone says nothing about it)
from sklearn.metrics import roc_auc_score

interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
in_scale, in_zp = inp["quantization"]
out_scale, out_zp = out["quantization"]
print(f"input  int8: scale={in_scale:.6g} per LSB, zero_point={in_zp}, shape={inp['shape']}")
print(f"output int8: scale={out_scale:.6g}, zero_point={out_zp}")

int8_probs = np.empty((len(X_test), len(class_names)), dtype=np.float32)
for i in range(len(X_test)):
    q = np.clip(np.round(X_test[i:i + 1] / in_scale + in_zp), -128, 127).astype(np.int8)
    interpreter.set_tensor(inp["index"], q)
    interpreter.invoke()
    int8_probs[i] = (interpreter.get_tensor(out["index"])[0].astype(np.float32) - out_zp) * out_scale

float_probs = cnn_1d.predict(X_test, batch_size=256, verbose=0)
print(f"Float32 macro AUC: {roc_auc_score(y_test, float_probs, average='macro'):.4f}")
print(f"INT8    macro AUC: {roc_auc_score(y_test, int8_probs, average='macro'):.4f}")
for k, name in enumerate(class_names):
    print(f"  {name:<5} float {roc_auc_score(y_test[:, k], float_probs[:, k]):.4f}"
          f" | int8 {roc_auc_score(y_test[:, k], int8_probs[:, k]):.4f}")
np.savez_compressed(RESULTS_DIR / "preds_1dcnn_int8.npz", cnn1d_int8=int8_probs)

input  int8: scale=0.05415 per LSB, zero_point=-25, shape=[   1 1000    1]
output int8: scale=0.00390625, zero_point=-128


Float32 macro AUC: 0.8302
INT8    macro AUC: 0.8294
  CD    float 0.8143 | int8 0.8122
  HYP   float 0.7935 | int8 0.7903
  MI    float 0.7846 | int8 0.7873
  NORM  float 0.8872 | int8 0.8873
  STTC  float 0.8714 | int8 0.8698
